# PRISM Theme 5 — FDB-v3 full evaluation (Kaggle, pristine fresh run)

## One-time setup checklist (before running)
1. **Settings (right panel) → Accelerator → GPU T4 x2** (or P100)
2. **Settings → Internet → ON** (needs a phone-verified account: Settings → Phone verification)
3. **Add-ons → Secrets → + Add secret** — add exactly these four labels:
   `LIVEKIT_URL` · `LIVEKIT_API_KEY` · `LIVEKIT_API_SECRET` · `GOOGLE_API_KEY`
4. Attach **no datasets** — this is a fresh run; the bundle is fetched from GitHub, the benchmark data from the official Drive link.

## Every past failure mode, pre-handled
| Past failure | Defense in this notebook |
|---|---|
| Agent worker never started (shell quoting bug) | quote-free supervisor loop, registration VERIFIED with hard stop |
| Wrong library version broke every tool call | **pinned `livekit-agents==1.8.3`** + version assertion + live tool-schema + positional-invocation test (cell 6) — hard stop before any hours spent |
| Empty results silently skipped by the batch | purge cell uses the `actual_tool_calls` JSON check (never file size) |
| Telemetry path drift → scores read as empty | agent and runner share one pinned path (PRISM_TOOL_LOG) |
| Duplicate calls hidden from the scored log | faithful per-execution logging |
| Runtime death mid-run losing results | checkpoint thread → `/kaggle/working/checkpoint` every 2 min; tip: "Save Version → Quick Save" occasionally |
| Rate-limit / transient failures | auto-retry of empty results, up to 3 passes |
| Credential mistakes | Kaggle Secrets — never typed in cells, never saved to the notebook |

## Running modes
- **Interactive:** keep the tab open; run cells in order.
- **Headless (recommended):** **Save Version → Save & Run All (Commit)** — survives browser close; watch progress from the version list. Secrets work in this mode.


In [ ]:
# @title 1. Get the validated bundle (dataset-first, GitHub fallback, integrity-checked)
import os, subprocess, zipfile
ZIP = "/kaggle/working/PRISM_Z_Colab.zip"
got = False
c = "/kaggle/input/prism-bundle/PRISM_Z_Colab.zip"
if os.path.exists(c):
    subprocess.run(["cp", c, ZIP], check=True)
    got = True
    print("bundle from:", c)
if not got:
    PINNED_COMMIT = "5b1b9a0"
    url = f"https://raw.githubusercontent.com/hamsavartn/Reflex-AI/{PINNED_COMMIT}/PRISM_Z_Colab.zip"
    r = subprocess.run(["wget", "-q", url, "-O", ZIP])
    got = r.returncode == 0 and os.path.exists(ZIP)
    print("bundle from GitHub raw:", "downloaded" if got else "FAILED")
try:
    with zipfile.ZipFile(ZIP) as z:
        bad = z.testzip()
        n = len(z.namelist())
    assert bad is None and n >= 8, f"corrupt bundle: bad={bad}, entries={n}"
    print(f"bundle integrity OK: {n} entries")
except Exception as e:
    msg = "BUNDLE INVALID - do not proceed. Details: " + str(e)
    msg += " | FIX: Data panel > + Add Input > Upload > New Dataset > drag PRISM_Z_Colab.zip from your PC > name it prism-bundle > attach > re-run."
    raise SystemExit(msg)
!unzip -q -o /kaggle/working/PRISM_Z_Colab.zip -d /kaggle/working/
!ls /kaggle/working/colab/prism_colab/v3/

In [ ]:
# @title 2. Install pinned stack (5–10 min)
import subprocess
!apt-get -qq install -y ffmpeg > /dev/null 2>&1
print("ffmpeg:", subprocess.run(["ffmpeg","-version"], capture_output=True, text=True).stdout.splitlines()[0][:45])
%cd /kaggle/working
!pip install -q "livekit-agents[google]==1.8.3" numpy python-dotenv pydub ffmpeg-python gdown
!pip install -q "nemo_toolkit[asr]"
import importlib.metadata as m
print("livekit-agents installed:", m.version("livekit-agents"))
print("stack installed")

In [ ]:
# @title 3. Unpack benchmark code + download data (fresh, ~750 MB)
import os, subprocess, glob, zipfile
os.chdir("/kaggle/working")
if not os.path.exists("/kaggle/working/Full-Duplex-Bench"):
    subprocess.run(["git","clone","--depth","1","https://github.com/DanielLin94144/Full-Duplex-Bench.git"], check=True)
src = "/kaggle/working/colab/prism_colab/v3"
dst = "/kaggle/working/Full-Duplex-Bench/v3"
subprocess.run(["cp","-r",src+"/.",dst+"/"], check=True)
os.makedirs(dst + "/logs", exist_ok=True)
if not os.path.exists(dst + "/fdb_v3_data_released"):
    subprocess.run(["gdown","1SO_4MTazWQ_jvCx0dtmpQ-t40bdd07yz","-O","/kaggle/working/fdb_v3_data.zip"], check=True)
    with zipfile.ZipFile("/kaggle/working/fdb_v3_data.zip") as z:
        z.extractall(dst)
n_wav = len(glob.glob(dst + "/fdb_v3_data_released/*/input.wav"))
print("examples with audio on disk:", n_wav, "(extra metadata-only dirs are ignored by the runner)")

In [ ]:
# @title 5. Credentials from Kaggle Secrets + quota sanity check
from kaggle_secrets import UserSecretsClient
import os
s = UserSecretsClient()
vals = {
    "LIVEKIT_URL": s.get_secret("LIVEKIT_URL"),
    "LIVEKIT_API_KEY": s.get_secret("LIVEKIT_API_KEY"),
    "LIVEKIT_API_SECRET": s.get_secret("LIVEKIT_API_SECRET"),
    "GOOGLE_API_KEY": s.get_secret("GOOGLE_API_KEY"),
}
os.makedirs("/kaggle/working/Full-Duplex-Bench/v3/logs", exist_ok=True)
env = "\n".join(f"{k}={v}" for k, v in vals.items()) + "\n"
open("/kaggle/working/Full-Duplex-Bench/v3/.env.local","w").write(env)
import google.genai as genai
c = genai.Client(api_key=vals["GOOGLE_API_KEY"])
r = c.models.generate_content(model="gemini-3.8-flash", contents="Reply with the word OK only.")
print("TEXT QUOTA:", r.text.strip()[:20], "OK")

In [ ]:
# @title 5. PRE-FLIGHT VERIFICATION — versions, tool schemas, positional invocation, duplicate guard
# This cell exists because of two past failures: version drift that broke every
# tool call, and wrapped tools losing positional arguments. If anything here
# fails, STOP — do not run the benchmark.
import asyncio, importlib.metadata as m, json, os, sys
va = m.version("livekit-agents")
print("livekit-agents:", va)
assert va == "1.8.3", f"VERSION DRIFT: expected 1.8.3, got {va} — fix cell 2 before proceeding"
os.chdir("/kaggle/working/Full-Duplex-Bench/v3")
sys.path.insert(0, os.getcwd())
import prism_agent
from livekit.agents import llm
from livekit.agents.llm.utils import function_arguments_to_pydantic_model

fnc = prism_agent.AssistantFnc(prism_agent.LatencyTracker(), "preflight-room")
tools = llm.find_function_tools(fnc)
names = sorted(t._info.name for t in tools)
print("tools registered:", len(tools))
assert len(tools) == 15, f"expected 15 tools, got {len(tools)}"

tt = [t for t in tools if t._info.name == "track_order"][0]
fields = list(function_arguments_to_pydantic_model(tt).model_fields.keys())
print("track_order schema params:", fields)
assert "order_id" in fields, "TOOL SCHEMA BROKEN — report to assistant, do not proceed"

async def preflight():
    # (a) positional invocation — the Oct-4 killer must stay dead
    try:
        res = await fnc.track_order("ABC123")
    except TypeError:
        res = await fnc.track_order(order_id="ABC123")  # callable API varies by patch version
    assert "order" in res.lower(), f"positional invocation broken: {res[:120]}"
    print("positional invocation: OK")
    # (b) duplicate state-change guard: identical booking twice must return the
    # SAME cached result (never a second booking)
    r1 = await fnc.book_flight("Preflight Tester")
    r2 = await fnc.book_flight("Preflight Tester")
    assert r1 == r2, f"DUPLICATE STATE CHANGE: {r1[:80]} vs {r2[:80]}"
    print("duplicate state-change guard: OK")
asyncio.run(preflight())
print("PRE-FLIGHT OK: versions, 15 tools, schema, positional invocation, duplicate guard verified")

In [ ]:
# @title 7. Start supervised worker + VERIFY registration
import subprocess, time, os
os.chdir("/kaggle/working/Full-Duplex-Bench/v3")
env = dict(os.environ, LK_PROVIDER="gemini2_5", HF_HOME="/kaggle/temp/hf", PRISM_GATE="0")
if os.path.exists("STOP_WORKER"): os.remove("STOP_WORKER")
sup = subprocess.Popen(
    'while [ ! -f STOP_WORKER ]; do python -u prism_agent.py start; echo sup_worker_restarted; sleep 3; done',
    shell=True, env=env, stdout=open("/kaggle/working/worker.log","w"), stderr=subprocess.STDOUT)
time.sleep(35)
log = open("/kaggle/working/worker.log").read()
if "registered worker" in log:
    print("WORKER REGISTERED: YES — proceed to cell 8")
else:
    print("WORKER REGISTERED: NO — send the output below to the assistant")
    print(log[-3000:])

In [ ]:
# @title 8. Run the 100-example benchmark (fresh; checkpointed every 2 min; NO --force)
import subprocess, os, glob, shutil, time, threading
os.chdir("/kaggle/working/Full-Duplex-Bench/v3")
env = dict(os.environ, HF_HOME="/kaggle/temp/hf", PRISM_GATE="0")
CK = "/kaggle/working/checkpoint"
os.makedirs(CK, exist_ok=True)
stop_ckpt = threading.Event()
def checkpointer():
    while not stop_ckpt.is_set():
        for f in glob.glob("fdb_v3_data_released/*/result_prism.json"):
            rel = os.path.relpath(f, ".")
            d = os.path.join(CK, os.path.dirname(rel))
            os.makedirs(d, exist_ok=True)
            shutil.copy(f, os.path.join(CK, rel))
        open("/kaggle/working/last_checkpoint.txt","w").write(time.strftime("%H:%M:%S"))
        stop_ckpt.wait(120)
threading.Thread(target=checkpointer, daemon=True).start()
total = len(glob.glob("fdb_v3_data_released/*/input.wav"))
!echo "--- done so far: $(ls fdb_v3_data_released/*/result_prism.json 2>/dev/null | wc -l)/$total ---   (first ~5 min are the ASR model download: silent)"
subprocess.run(["python","-u","run_tool_benchmark_all_released.py","--provider","prism","--root_dir","fdb_v3_data_released"], env=env)
stop_ckpt.set(); time.sleep(2)
done = len(glob.glob("fdb_v3_data_released/*/result_prism.json"))
print(f"=== BATCH COMPLETE: {done}/{total} results ===")

In [ ]:
# @title 9. Auto-retry empty results (up to 3 passes — rate limits, worker restarts)
import subprocess, os, glob, json
os.chdir("/kaggle/working/Full-Duplex-Bench/v3")
env = dict(os.environ, HF_HOME="/kaggle/temp/hf", PRISM_GATE="0")
def empties():
    out = []
    for f in glob.glob("fdb_v3_data_released/*/result_prism.json"):
        if not json.load(open(f, encoding="utf-8")).get("actual_tool_calls"):
            out.append(f)
    return out
for attempt in range(3):
    e = empties()
    print(f"pass {attempt+1}: {len(e)} empty results")
    if not e:
        break
    for f in e:
        os.remove(f)
    subprocess.run(["python","-u","run_tool_benchmark_all_released.py","--provider","prism","--root_dir","fdb_v3_data_released"], env=env)
print("FINAL empty count:", len(empties()))

In [ ]:
# @title 10. Evaluate (tool F1, pass rate, latency)
import subprocess, os, json
os.chdir("/kaggle/working/Full-Duplex-Bench/v3")
B = "--benchmark benchmark_data_v2.json --results-dir fdb_v3_data_released --provider prism"
subprocess.run(f"python evaluate_tool_calls.py {B} --output prism_evaluation_report.json", shell=True)
subprocess.run(f"python evaluate_pass_rate.py {B} --output prism_pass_rate_report.json", shell=True)
subprocess.run(f"python analyze_tool_latency.py --results-dir fdb_v3_data_released --provider prism", shell=True)
r = json.load(open("prism_pass_rate_report.json"))
print("\n=== STRICT PASS RATE:", r["overall_pass_rate"], f"({r['passed']}/{r['total_scenarios']}) ===")

In [ ]:
# @title 11. Package results into the notebook Output
import zipfile, os, glob
os.chdir("/kaggle/working/Full-Duplex-Bench/v3")
with zipfile.ZipFile("/kaggle/working/prism_results.zip","w",zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("fdb_v3_data_released/*/result_prism.json"):
        z.write(f)
    for f in glob.glob("prism_*.json") + glob.glob("logs/*.log"):
        z.write(f)
print("size MB:", round(os.path.getsize("/kaggle/working/prism_results.zip")/1e6, 1))
print("Download prism_results.zip from the notebook Output panel (right side).")